# FittsLab — ML Training Notebook
**Bachelor's Thesis: Neural Network vs. Fitts' Law for Mouse Movement Time Prediction**

This notebook:
1. Loads exported FittsLab experiment data (CSV)
2. Trains a neural network using Keras/TensorFlow
3. Calculates Fitts' Law baseline predictions
4. Evaluates both models (MAE, RMSE, R²)
5. Exports `predictions.csv` for import into FittsLab app

**Input**: `fittslab_results_YYYY-MM-DD.csv` (from FittsLab CSV export)  
**Output**: `predictions.csv` (import into ML Vorhersage tab)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import warnings
warnings.filterwarnings('ignore')

print(f"TensorFlow version: {tf.__version__}")
np.random.seed(42)
tf.random.set_seed(42)

## 1. Daten laden & vorbereiten

In [ ]:
import glob
import os

# Load the most recent FittsLab CSV export
csv_files = glob.glob('fittslab_results_*.csv')
if not csv_files:
    raise FileNotFoundError(
        "Keine fittslab_results_*.csv gefunden!\n"
        "Bitte zuerst ein Experiment in FittsLab durchführen und als CSV exportieren."
    )

# Use most recent file
csv_file = sorted(csv_files)[-1]
print(f"Lade Datei: {csv_file}")

# Read only the data rows (stop before Summary section)
raw = open(csv_file, encoding='utf-8').read()
data_section = raw.split('\n\nSummary')[0] if '\n\nSummary' in raw else raw

from io import StringIO
df_raw = pd.read_csv(StringIO(data_section))

# Keep only HIT rows with valid data
df = df_raw[df_raw['Hit_Status'] == 'HIT'].copy()
df = df[['Distance_px', 'Width_px', 'ID_bits', 'Actual_MT_ms']].dropna()
df = df[df['Actual_MT_ms'] > 0]  # Remove invalid times
df = df[df['ID_bits'] > 0]       # Remove invalid ID values
df.columns = ['Distance_px', 'Width_px', 'ID_bits', 'Measured_MT_ms']
df = df.reset_index(drop=True)

print(f"Geladene Samples: {len(df)}")
print(f"\nStatistiken:")
print(df.describe().round(2))

## 2. Explorative Datenanalyse (EDA)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].scatter(df['ID_bits'], df['Measured_MT_ms'], alpha=0.6, color='black', s=30)
axes[0].set_xlabel('Index of Difficulty (bits)')
axes[0].set_ylabel('Movement Time (ms)')
axes[0].set_title('ID vs. MT (Rohdaten)')

axes[1].hist(df['Measured_MT_ms'], bins=20, color='steelblue', edgecolor='white')
axes[1].set_xlabel('Movement Time (ms)')
axes[1].set_title('MT Verteilung')

axes[2].scatter(df['Distance_px'], df['Measured_MT_ms'], alpha=0.6, color='darkblue', s=30)
axes[2].set_xlabel('Distanz (px)')
axes[2].set_ylabel('MT (ms)')
axes[2].set_title('Distanz vs. MT')

plt.tight_layout()
plt.savefig('eda_plots.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Korrelation ID ↔ MT: {df['ID_bits'].corr(df['Measured_MT_ms']):.3f}")

## 3. Fitts' Law Baseline

Klassische lineare Regression: MT = a + b × ID

In [ ]:
from numpy.polynomial import polynomial as P

# Fit linear regression: MT = a + b * ID
coeffs = np.polyfit(df['ID_bits'], df['Measured_MT_ms'], 1)
b_fit, a_fit = coeffs
print(f"Fitts' Law gefittet: MT = {a_fit:.1f} + {b_fit:.1f} × ID")

df['FittsLaw_MT_ms'] = a_fit + b_fit * df['ID_bits']

# Metrics
mae_fitts  = mean_absolute_error(df['Measured_MT_ms'], df['FittsLaw_MT_ms'])
rmse_fitts = np.sqrt(mean_squared_error(df['Measured_MT_ms'], df['FittsLaw_MT_ms']))
r2_fitts   = r2_score(df['Measured_MT_ms'], df['FittsLaw_MT_ms'])

print(f"\nFitts' Law Baseline:")
print(f"  MAE  = {mae_fitts:.1f} ms")
print(f"  RMSE = {rmse_fitts:.1f} ms")
print(f"  R²   = {r2_fitts:.4f}")

## 4. Neural Network Training

Input features: Distance (px), Width (px)  
Target: Movement Time (ms)

In [ ]:
X = df[['Distance_px', 'Width_px']].values
y = df['Measured_MT_ms'].values

if len(df) < 10:
    raise ValueError(f"Zu wenig Daten ({len(df)} Samples). Mindestens 10 HITs für Training benötigt.")

test_size = 0.2 if len(df) >= 20 else 0.1
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, random_state=42)

scaler_X = StandardScaler()
X_train_s = scaler_X.fit_transform(X_train)
X_test_s  = scaler_X.transform(X_test)

scaler_y = StandardScaler()
y_train_s = scaler_y.fit_transform(y_train.reshape(-1, 1)).ravel()

print(f"Training samples: {len(X_train)}")
print(f"Test samples:     {len(X_test)}")

In [ ]:
model = keras.Sequential([
    layers.Input(shape=(2,)),
    layers.Dense(64, activation='relu'),
    layers.Dense(64, activation='relu'),
    layers.Dense(32, activation='relu'),
    layers.Dense(1)
], name='FittsNN')

model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.001), loss='mae')
model.summary()

early_stop = keras.callbacks.EarlyStopping(monitor='val_loss', patience=30, restore_best_weights=True)

history = model.fit(
    X_train_s, y_train_s,
    validation_split=0.2,
    epochs=300,
    batch_size=max(4, len(X_train) // 8),
    callbacks=[early_stop],
    verbose=0
)

print(f"Training beendet nach {len(history.history['loss'])} Epochen")

# Plot training history
plt.figure(figsize=(8, 4))
plt.plot(history.history['loss'], label='Training Loss', color='steelblue')
plt.plot(history.history['val_loss'], label='Validation Loss', color='red', linestyle='--')
plt.xlabel('Epoche')
plt.ylabel('MAE (normalisiert)')
plt.title('Training History')
plt.legend()
plt.tight_layout()
plt.savefig('training_history.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Predict on all data for the comparison CSV
X_all_s = scaler_X.transform(X)
y_pred_s = model.predict(X_all_s, verbose=0).ravel()
y_pred   = scaler_y.inverse_transform(y_pred_s.reshape(-1, 1)).ravel()

df['NeuralNet_MT_ms'] = y_pred

# Evaluate on test set
X_test_pred_s = model.predict(X_test_s, verbose=0).ravel()
X_test_pred   = scaler_y.inverse_transform(X_test_pred_s.reshape(-1, 1)).ravel()

mae_nn  = mean_absolute_error(y_test, X_test_pred)
rmse_nn = np.sqrt(mean_squared_error(y_test, X_test_pred))
r2_nn   = r2_score(y_test, X_test_pred)

print("=" * 40)
print(f"{'Metric':<10} {'Fitts Law':>12} {'Neural Net':>12}")
print("=" * 40)
print(f"{'MAE (ms)':<10} {mae_fitts:>12.1f} {mae_nn:>12.1f}")
print(f"{'RMSE (ms)':<10} {rmse_fitts:>12.1f} {rmse_nn:>12.1f}")
print(f"{'R²':<10} {r2_fitts:>12.4f} {r2_nn:>12.4f}")
print("=" * 40)

winner = "Neural Network" if mae_nn < mae_fitts else "Fitts' Law"
improvement = abs(mae_fitts - mae_nn) / mae_fitts * 100
print(f"\nBesseres Modell: {winner} (Δ MAE = {improvement:.1f}%)")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter: ID vs MT
id_range = np.linspace(df['ID_bits'].min(), df['ID_bits'].max(), 100)
fitts_line = a_fit + b_fit * id_range

axes[0].scatter(df['ID_bits'], df['Measured_MT_ms'], color='black', alpha=0.6, s=30, label='Gemessen', zorder=3)
axes[0].plot(id_range, fitts_line, color='red', linestyle='--', linewidth=2, label="Fitts' Law")
sorted_idx = np.argsort(df['ID_bits'].values)
axes[0].plot(df['ID_bits'].values[sorted_idx], df['NeuralNet_MT_ms'].values[sorted_idx], color='blue', linewidth=2, label='Neural Network')
axes[0].set_xlabel('Index of Difficulty (bits)')
axes[0].set_ylabel('Movement Time (ms)')
axes[0].set_title('ID vs. MT — Modellvergleich')
axes[0].legend()

# Error comparison bar
models = ["Fitts' Law", 'Neural Network']
maes   = [mae_fitts, mae_nn]
colors = ['red' if mae_fitts > mae_nn else 'green', 'blue' if mae_nn < mae_fitts else 'orange']
bars = axes[1].bar(models, maes, color=['#ef4444', '#2563eb'], alpha=0.8, edgecolor='white')
axes[1].set_ylabel('MAE (ms)')
axes[1].set_title('Modell-Vergleich: MAE')
for bar, val in zip(bars, maes):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2, f'{val:.1f}ms', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Export — predictions.csv

Diese Datei in FittsLab → "ML Vorhersage" Tab importieren.

In [ ]:
export_df = df[['Distance_px', 'Width_px', 'ID_bits', 'Measured_MT_ms', 'FittsLaw_MT_ms', 'NeuralNet_MT_ms']].copy()
export_df.insert(0, 'Sample', range(1, len(export_df) + 1))
export_df = export_df.round(3)

output_file = 'predictions.csv'
export_df.to_csv(output_file, index=False)

print(f"Exportiert: {output_file}  ({len(export_df)} Samples)")
print(f"\nVorschau:")
print(export_df.head(5).to_string(index=False))
print(f"\nNächster Schritt: '{output_file}' in FittsLab → 'ML Vorhersage' Tab importieren.")